# TheTelos — Sesli Özet Üretici (Kokoro)

1. Üst menü **Çalışma zamanı → Çalışma zamanı türünü değiştir → T4 GPU** seçin.
2. Soldaki **🔑 (Gizli anahtarlar / Secrets)** paneline şu 5 değeri ekleyin ve her birinde "Not defteri erişimi" açık olsun:
   - `TLS_KEY` — paneldeki *Sesli Özet* sayfasındaki anahtar
   - `R2_ACCOUNT_ID`, `R2_ACCESS_KEY`, `R2_SECRET` — Cloudflare R2 API anahtarı
   - `R2_BUCKET` — kova adı (ör. `thetelos-audio`)
3. Aşağıdaki hücreleri sırayla çalıştırın (▶). Son hücre kitapları tek tek seslendirip siteye bağlar; durdurup sonra tekrar çalıştırabilirsiniz, kaldığı yerden devam eder.

In [ ]:
!apt-get -qq -y install espeak-ng > /dev/null
!pip install -q "kokoro>=0.9.4" soundfile boto3 requests

In [ ]:
# AYARLAR
SITE        = "https://thetelos.org/thetelos-panel/api/tts.php"
PUBLIC_BASE = "https://audio.thetelos.org"   # R2 kovasının herkese açık adresi (panelde yazan)
VOICE       = "am_michael"                     # erkek, sakin anlatıcı. Kadın: "af_heart"
SPEED       = 1.0

from google.colab import userdata
TLS_KEY    = userdata.get("TLS_KEY")
R2_ACCOUNT = userdata.get("R2_ACCOUNT_ID")
R2_KEY     = userdata.get("R2_ACCESS_KEY")
R2_SECRET  = userdata.get("R2_SECRET")
R2_BUCKET  = userdata.get("R2_BUCKET")
print("Ayarlar tamam")

In [ ]:
import requests, boto3, subprocess, numpy as np, soundfile as sf, time, os
from kokoro import KPipeline

pipe = KPipeline(lang_code="a")   # Amerikan İngilizcesi
s3 = boto3.client("s3", endpoint_url=f"https://{R2_ACCOUNT}.r2.cloudflarestorage.com",
                  aws_access_key_id=R2_KEY, aws_secret_access_key=R2_SECRET, region_name="auto")

def speak(text):
    parts = []
    for _, _, audio in pipe(text, voice=VOICE, speed=SPEED, split_pattern=r"\n+"):
        if audio is not None:
            parts.append(audio.numpy() if hasattr(audio, "numpy") else audio)
            parts.append(np.zeros(int(24000 * 0.35), dtype=np.float32))   # paragraf arası kısa nefes
    return np.concatenate(parts) if parts else None

def run(max_books=100000):
    done = 0
    while done < max_books:
        r = requests.get(SITE, params={"action": "list", "key": TLS_KEY, "limit": 5}, timeout=120).json()
        items = r.get("items", [])
        if not items:
            print("Seslendirilecek kitap kalmadı ✓"); break
        for it in items:
            t0 = time.time(); pid = it["post_id"]
            audio = speak(it["text"])
            if audio is None:
                print("boş ses, atlandı:", pid); continue
            sf.write("/tmp/a.wav", audio, 24000)
            subprocess.run(["ffmpeg", "-y", "-loglevel", "error", "-i", "/tmp/a.wav", "-ac", "1", "-b:a", "64k", "/tmp/a.mp3"], check=True)
            key = f"audio/{pid}.mp3"
            s3.upload_file("/tmp/a.mp3", R2_BUCKET, key, ExtraArgs={"ContentType": "audio/mpeg", "CacheControl": "public, max-age=31536000"})
            secs = int(len(audio) / 24000)
            a = requests.post(SITE, data={"action": "attach", "key": TLS_KEY, "post_id": pid, "url": f"{PUBLIC_BASE}/{key}", "seconds": secs}, timeout=60).json()
            done += 1
            print(f"✓ {done}. {it['title'][:60]} — {secs//60} dk ses, {int(time.time()-t0)} sn'de üretildi", "" if a.get("ok") else a)

run()